In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
from matplotlib import font_manager

# Font setup with fallback to sans-serif if the path is missing.
font_path = '../../data/Arial.ttf'
if os.path.exists(font_path):
    font_manager.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'Arial'
else:
    plt.rcParams['font.family'] = 'sans-serif' # Fallback

In [ ]:
df = pd.read_csv('../../data/processed/fig/fig1b_sdg_r2_compare_token.csv')
df

In [ ]:
# Configuration.
plot_cols = ['spatial_self_r2', 'imagenet_r2', 'seg_r2']
column_labels = ['$R^2$ (Ours)', '$R^2$ (ImageNet)', '$R^2$ (Segmentation)']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

In [ ]:
# Maximum value per row.
max_values = df[plot_cols].max(axis=1)

# Normalize each row by its max (handles div-by-zero).
df_norm = df.copy()
df_norm[plot_cols] = df_norm[plot_cols].div(max_values, axis=0).fillna(0)

# Prepare plotting data.
labels = df_norm['SDG'].tolist()
num_vars = len(labels)

# Angles: clockwise, starting from 12 o'clock.
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1] # close the polygon

# --- 3. Plot ---

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True), dpi=300)

# Radar orientation: start at 12 o'clock, clockwise.
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Grid style.
ax.yaxis.grid(True, color='#D3D3D3', linestyle='--', linewidth=0.7)
ax.xaxis.grid(True, color='#D3D3D3', linestyle='--', linewidth=0.7)

# --- 4. Axes + labels ---

# X-axis (category labels).
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11, fontweight='medium')

# Adjust label placement to avoid overlap.
for label, angle in zip(ax.get_xticklabels(), angles[:-1]):
    x, y = label.get_position()
    # Convert angle to degrees for branching.
    angle_deg = np.degrees(angle)
    if angle_deg == 0 or angle_deg == 180:
        label.set_horizontalalignment('center')
    elif 0 < angle_deg < 180:
        label.set_horizontalalignment('left') # right half: left-align labels
    else:
        label.set_horizontalalignment('right') # left half: right-align labels

# Y-axis (radial) setup.
ax.set_ylim(0, 1.15) # leave headroom for top labels
ax.set_yticklabels([]) # hide default ticks
ax.set_yticks([0.5, 1.0]) # keep only mid and outer rings

# --- 5. Plot data ---

for i, col in enumerate(plot_cols):
    values = df_norm[col].tolist()
    values += values[:1]
    
    # Plot the line.
    ax.plot(angles, values, linewidth=2, color=colors[i], label=column_labels[i], zorder=10)
    # Plot the filled area.
    ax.fill(angles, values, color=colors[i], alpha=0.1, zorder=1)
    # Plot the data points.
    ax.scatter(angles, values, facecolors=colors[i], edgecolors='white', 
               s=60, linewidths=1.5, zorder=11)

# --- 6. Annotate true values (key polish) ---

# Annotate the true value at axis top (1.0) and mid (0.5).
for i, angle in enumerate(angles[:-1]):
    real_max = max_values.iloc[i]
    
    # Top label (max value), bold.
    ax.text(angle, 1.18, f"{real_max:.2f}", 
            horizontalalignment='center', verticalalignment='center',
            fontsize=9, fontweight='bold', color='black', zorder=20)
    
    # Mid label (50% value), gray so it doesn't overpower.
    # Background box so grid lines don't obscure text.
    ax.text(angle, 0.5, f"{real_max * 0.5:.2f}", 
            horizontalalignment='center', verticalalignment='center',
            fontsize=7.5, color='#555555', zorder=15,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=0.5))

# --- 7. Legend + layout ---

# Place legend at bottom or corner to avoid crowding the figure.
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), 
          ncol=3, frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig('../../data/figure_assets/fig1_radar.svg', format='svg', bbox_inches='tight')
plt.show()